In [2]:

import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score



In [3]:
print("Inisialisasi Data")

data = {
    'Temperatur': [10, 25, 15, 20, 18, 20, 22, 24],
    'Angin': [0, 0, 5, 3, 7, 10, 5, 6],
    'Klasifikasi': ['Dingin', 'Panas', 'Dingin', 'Panas', 'Dingin', 'Dingin', 'Panas', 'Panas']
}
df = pd.DataFrame(data)

# Variabel fitur (X) dan target (y)
X = df[['Temperatur', 'Angin']].values
y = df['Klasifikasi'].values

# Data Test yang akan diklasifikasikan (16°C, 3 km/jam)
data_test = np.array([[16, 3]]) 

print(df)


Inisialisasi Data
   Temperatur  Angin Klasifikasi
0          10      0      Dingin
1          25      0       Panas
2          15      5      Dingin
3          20      3       Panas
4          18      7      Dingin
5          20     10      Dingin
6          22      5       Panas
7          24      6       Panas


In [4]:
print("Perhitungan Jarak dan Urutan Tetangga")
print("Data Test: (Temperatur=16, Angin=3)")

# Hitung Jarak Euclidean (secara manual)
jarak_list = []
for i in range(len(X)):
    # Jarak = sqrt((T_data - T_test)^2 + (V_data - V_test)^2)
    jarak = np.sqrt(np.sum((data_test[0] - X[i])**2))
    jarak_list.append((jarak, y[i]))

# Urutkan berdasarkan jarak terdekat
jarak_list.sort(key=lambda x: x[0])

# Tampilkan hasil jarak yang sudah diurutkan
print("\nUrutan Jarak Terdekat:")
for i, (jarak, klasifikasi) in enumerate(jarak_list):
    print(f"Rank {i+1}: Jarak={jarak:.2f}, Klasifikasi={klasifikasi}")

### 2. Perhitungan Jarak dan Urutan Tetangga ###
Data Test: (Temperatur=16, Angin=3)

Urutan Jarak Terdekat:
Rank 1: Jarak=2.24, Klasifikasi=Dingin
Rank 2: Jarak=4.00, Klasifikasi=Panas
Rank 3: Jarak=4.47, Klasifikasi=Dingin
Rank 4: Jarak=6.32, Klasifikasi=Panas
Rank 5: Jarak=6.71, Klasifikasi=Dingin
Rank 6: Jarak=8.06, Klasifikasi=Dingin
Rank 7: Jarak=8.54, Klasifikasi=Panas
Rank 8: Jarak=9.49, Klasifikasi=Panas


In [5]:
print("Klasifikasi Data Test dengan k=3 ")

k_val = 3
tetangga_k3 = jarak_list[:k_val]

# Hitung voting dari 3 tetangga terdekat
voting_k3 = {}
for jarak, klasifikasi in tetangga_k3:
    voting_k3[klasifikasi] = voting_k3.get(klasifikasi, 0) + 1

# Tentukan hasil prediksi (mayoritas)
hasil_prediksi = max(voting_k3, key=voting_k3.get)

print(f"Tetangga (k={k_val}) yang diambil:")
for jarak, klasifikasi in tetangga_k3:
    print(f"- Klasifikasi: {klasifikasi} (Jarak: {jarak:.2f})")

print(f"\nHasil Voting: {voting_k3}")
print(f"Klasifikasi Data Test (k=3) adalah: **{hasil_prediksi}**")

Klasifikasi Data Test dengan k=3 
Tetangga (k=3) yang diambil:
- Klasifikasi: Dingin (Jarak: 2.24)
- Klasifikasi: Panas (Jarak: 4.00)
- Klasifikasi: Dingin (Jarak: 4.47)

Hasil Voting: {'Dingin': 2, 'Panas': 1}
Klasifikasi Data Test (k=3) adalah: **Dingin**


In [6]:
print("Menentukan Nilai k Terbaik (LOOCV)")

loo = LeaveOneOut()
k_values = [1, 3, 5, 7] # Nilai k ganjil yang akan diuji
results = {}

for k in k_values:
    y_true_loo = []
    y_pred_loo = []
    
    # Inisialisasi model K-NN
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Loop LOOCV: Latih pada N-1 data, Uji pada 1 data
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        knn.fit(X_train, y_train)
        y_pred = knn.predict(X_test)
        
        y_true_loo.extend(y_test)
        y_pred_loo.extend(y_pred)

    # Hitung akurasi
    accuracy = accuracy_score(y_true_loo, y_pred_loo)
    results[k] = accuracy
    print(f"Akurasi untuk k={k}: {accuracy:.4f} ({int(accuracy * len(X))}/{len(X)} Benar)")

# Tentukan k terbaik
best_k = max(results, key=results.get)
best_accuracy = results[best_k]

print(f"\nNilai k terbaik berdasarkan akurasi tertinggi: **k={best_k}** (Akurasi: {best_accuracy:.4f})")

Menentukan Nilai k Terbaik (LOOCV)
Akurasi untuk k=1: 1.0000 (8/8 Benar)
Akurasi untuk k=3: 0.7500 (6/8 Benar)
Akurasi untuk k=5: 0.7500 (6/8 Benar)
Akurasi untuk k=7: 0.0000 (0/8 Benar)

Nilai k terbaik berdasarkan akurasi tertinggi: **k=1** (Akurasi: 1.0000)
